
# 🏠 LASSO on Kaggle House Prices — Simple, Step‑by‑Step

This notebook shows a **clean, beginner‑friendly** LASSO workflow on Kaggle’s **House Prices** dataset.

**Before you run (Kaggle):**
1. Click **Add data** → add **“House Prices - Advanced Regression Techniques”** (competition).
2. Run this notebook top to bottom.

**Locally:** download the data and update `DATA_DIR` below.


In [ ]:

# =========================
# Full workflow in one cell
# =========================
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Detect Kaggle path; else set local folder
DEFAULT_KAGGLE = Path('/kaggle/input/house-prices-advanced-regression-techniques')
DATA_DIR = DEFAULT_KAGGLE if DEFAULT_KAGGLE.exists() else Path('path/to/your/downloaded/folder')

# Load data
df = pd.read_csv(DATA_DIR / 'train.csv')
target_col = 'SalePrice'
y = np.log1p(df[target_col].copy())
X = df.drop(columns=[target_col, 'Id'])

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Numeric features: {len(numeric_cols)}, Categorical features: {len(categorical_cols)}")

# Preprocessing
numeric_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                   ('scaler', StandardScaler())])
categorical_pipeline = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                                       ('ohe', OneHotEncoder(handle_unknown='ignore', sparse=False))])
preprocessor = ColumnTransformer([('num', numeric_pipeline, numeric_cols),
                                  ('cat', categorical_pipeline, categorical_cols)],
                                 remainder='drop')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE)

def get_feature_names(preproc):
    try:
        return preproc.get_feature_names_out()
    except Exception:
        cat_steps = preproc.named_transformers_['cat']['ohe']
        cat_ohe_names = cat_steps.get_feature_names_out(categorical_cols)
        return np.array(list(numeric_cols) + list(cat_ohe_names))

# Baselines
alpha_grid = np.logspace(-4, 2, 50)
models = {
    'OLS (no regularization)': Pipeline([('preprocess', preprocessor), ('model', LinearRegression())]),
    'Ridge (CV)': Pipeline([('preprocess', preprocessor), ('model', RidgeCV(alphas=alpha_grid, cv=5))]),
    'LASSO (CV)': Pipeline([('preprocess', preprocessor), ('model', LassoCV(alphas=alpha_grid, cv=5, max_iter=20000, random_state=RANDOM_STATE))])
}

results, fitted = [], {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    rmse = mean_squared_error(y_test, preds, squared=False)
    results.append((name, rmse))
    fitted[name] = pipe
    print(f"{name} RMSE (log-price): {rmse:.4f}")

print("\nBaseline comparison (lower RMSE is better):")
for name, rmse in sorted(results, key=lambda x: x[1]):
    print(f" - {name}: {rmse:.4f}")

lasso_cv = fitted['LASSO (CV)'].named_steps['model']
print(f"\nBest alpha chosen by LASSO (CV): {lasso_cv.alpha_:.6f}")

# What did LASSO keep?
preprocessor_fitted = fitted['LASSO (CV)'].named_steps['preprocess']
X_train_pre = preprocessor_fitted.transform(X_train)
feature_names = get_feature_names(preprocessor_fitted)

lasso_coef = lasso_cv.coef_
importances = pd.DataFrame({'feature': feature_names, 'coef': lasso_coef})
kept = importances[importances.coef != 0]
print(f"\nLASSO kept {kept.shape[0]} features out of {importances.shape[0]}.")
display(kept.reindex(kept['coef'].abs().sort_values(ascending=False).index).head(15))

# Push further A: sparsity & performance paths
alphas_for_path = np.logspace(-4, 1, 15)
nonzero_counts, rmses = [], []
X_test_pre = preprocessor_fitted.transform(X_test)

for a in alphas_for_path:
    model = Lasso(alpha=a, max_iter=20000, random_state=RANDOM_STATE).fit(X_train_pre, y_train)
    preds = model.predict(X_test_pre)
    nonzero_counts.append(np.count_nonzero(model.coef_))
    rmses.append(mean_squared_error(y_test, preds, squared=False))

plt.figure(figsize=(6,4))
plt.plot(alphas_for_path, nonzero_counts, marker='o')
plt.xscale('log')
plt.xlabel('alpha (bigger = stronger penalty)')
plt.ylabel('# non-zero coefficients')
plt.title('LASSO sparsity path')
plt.show()

plt.figure(figsize=(6,4))
plt.plot(alphas_for_path, rmses, marker='o')
plt.xscale('log')
plt.xlabel('alpha')
plt.ylabel('RMSE (log-price)')
plt.title('Performance vs regularization')
plt.show()

# Push further B: stability selection (bootstrap)
N_BOOTSTRAPS = 20  # keep modest for Kaggle speed
selected_counts = np.zeros_like(lasso_coef, dtype=float)
name_to_pos_main = {n:i for i,n in enumerate(feature_names)}

for b in range(N_BOOTSTRAPS):
    idx = np.random.choice(len(X_train), size=len(X_train), replace=True)
    Xb = X_train.iloc[idx]
    yb = y_train.iloc[idx]
    p_boot = ColumnTransformer([('num', numeric_pipeline, numeric_cols),
                                ('cat', categorical_pipeline, categorical_cols)],
                               remainder='drop').fit(Xb)
    Xb_pre = p_boot.transform(Xb)
    fb_names = p_boot.get_feature_names_out()
    lasso_b = Lasso(alpha=lasso_cv.alpha_, max_iter=20000, random_state=RANDOM_STATE).fit(Xb_pre, yb)
    sel_mask = (lasso_b.coef_ != 0)
    for n, is_sel in zip(fb_names, sel_mask):
        if is_sel and n in name_to_pos_main:
            selected_counts[name_to_pos_main[n]] += 1

stability = pd.DataFrame({'feature': feature_names,
                          'selection_freq': selected_counts / N_BOOTSTRAPS}).sort_values('selection_freq', ascending=False)
display(stability.head(15))

plt.figure(figsize=(6,4))
plt.barh(stability.head(15)['feature'][::-1], stability.head(15)['selection_freq'][::-1])
plt.xlabel('Selection frequency (0–1)')
plt.title('Top 15 stable features (LASSO)')
plt.tight_layout()
plt.show()

# Random Forest comparison
rf_pipe = Pipeline([('preprocess', preprocessor),
                    ('rf', RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1))])
rf_pipe.fit(X_train, y_train)
rf_rmse = mean_squared_error(y_test, rf_pipe.predict(X_test), squared=False)
print(f"\nRandom Forest RMSE (log-price): {rf_rmse:.4f}")

rf_pre = rf_pipe.named_steps['preprocess']
rf_names = rf_pre.get_feature_names_out()
rf_importances = rf_pipe.named_steps['rf'].feature_importances_
rf_top = pd.DataFrame({'feature': rf_names, 'importance': rf_importances}).sort_values('importance', ascending=False).head(15)
display(rf_top)



## 9) Bonus — Elastic Net (mix of L1 and L2)
**Why?** Elastic Net combines **LASSO (L1)** and **Ridge (L2)**.  
- L1 gives **feature selection** (sparsity).  
- L2 helps when features are **correlated**, stabilizing coefficients.

We’ll try multiple `l1_ratio` values and pick the best via cross‑validation.


In [ ]:

from sklearn.linear_model import ElasticNetCV

# Try a grid of alphas and l1_ratios
alpha_grid = np.logspace(-4, 2, 50)
l1_grid = [0.2, 0.5, 0.8, 0.95]

enet_pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model', ElasticNetCV(alphas=alpha_grid, l1_ratio=l1_grid, cv=5, max_iter=20000, random_state=RANDOM_STATE))
])

enet_pipe.fit(X_train, y_train)
enet_preds = enet_pipe.predict(X_test)
enet_rmse = mean_squared_error(y_test, enet_preds, squared=False)

enet_cv = enet_pipe.named_steps['model']
print(f"Elastic Net RMSE (log-price): {enet_rmse:.4f}")
print(f"Best alpha: {enet_cv.alpha_:.6f}, Best l1_ratio: {enet_cv.l1_ratio_:.2f}")

# Which features are kept (non-zero)?
enet_pre = enet_pipe.named_steps['preprocess']
enet_names = enet_pre.get_feature_names_out()
enet_coefs = enet_cv.coef_

enet_keep = pd.DataFrame({'feature': enet_names, 'coef': enet_coefs})
enet_keep = enet_keep[enet_keep['coef'] != 0].sort_values('coef', key=lambda s: s.abs(), ascending=False)

print(f"Elastic Net kept {enet_keep.shape[0]} features.")
display(enet_keep.head(15))
